# Introduction to Vector Stores


- Use Langchain documentation to create vector stores using chromadb, milvus, weaviate, pinecone: https://python.langchain.com/docs/integrations/vectorstores/
- Create embeddings of a document and index into vector stores: https://huggingface.co/blog/getting-started-with-embeddings
- For the task you will be indexing the book: https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf
- Experiment different vector stores and other variables for and present your findings for optimizing retrieval (the most appropriate parts should be returned for any query)
- Hint: chunking

**Steps**
- Go through the given readings. Take an hour at max.
- Index document as vectors. Use any documentation to help you, but avoid using AI Tools.
- For embedding model, use an open source model from huggingface.
- Query should return most appropriate parts in relevance to the query. *Remember, your task is to build effective retrieval*


**Resources for understanding vector search**
- https://weaviate.io/blog/vector-search-explained


**Additional resources for understanding embeddings**
- https://cohere.com/llmu/text-embeddings
- https://docs.cohere.com/v2/docs/embeddings
- https://docs.cohere.com/v2/docs/playground-overview

In [3]:
!pip install -qU langchain-google-genai

**Creating Vector Store**

In [41]:
import getpass
import os

if not os.environ.get("GOOGLE_API_KEY"):
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

Enter API key for Google Gemini: ··········


In [32]:
!pip install langchain_community pypdf langchain_text_splitters

In [30]:
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF
loader = PyPDFLoader("harry-potter-and-the-philosophers-stone-by-jk-rowling.pdf")

# Load pages
pages = loader.load()

# Print first page's content
print(pages[0])


page_content='' metadata={'producer': 'Acrobat Distiller 7.0.5 (Windows)', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2012-05-01T23:57:27+10:00', 'subject': "When a letter arrives for unhappy but ordinary Harry Potter, a decade-old secret is revealed to him. His parents were wizards, killed by a Dark Lord's curse when Harry was just a baby, and which he somehow survived. Escaping from his unbearable Muggle guardians to Hogwarts, a wizarding school brimming with ghosts and enchantments, Harry stumbles into a sinister adventure when he finds a threeheaded dog guarding a room on the third floor. Then he hears of a missing stone with astonishing powers which could be valuable, dangerous, or both.", 'author': 'J.K. Rowling', 'keywords': '', 'moddate': '2012-07-01T14:20:39+10:00', 'title': 'Harry Potter and the Philosopher’s Stone', 'source': 'harry-potter-and-the-philosophers-stone-by-jk-rowling.pdf', 'total_pages': 228, 'page': 0, 'page_label': '1'}


In [35]:
len(pages[15].page_content)

2263

In [36]:
from langchain_text_splitters import CharacterTextSplitter

splitter=CharacterTextSplitter(separator='\n', chunk_size=500, chunk_overlap=50)


splitted_pages=splitter.split_documents(pages)


In [40]:
print(splitted_pages[15].page_content)

8 Harry Potter  
 
happily as she wrestled a screaming Dudley into his high chair. 
None of them noticed a large tawny owl flutter past the window. 
At half past eight, Mr Dursley picked up his briefcase, pecked 
Mrs Dursley on the cheek and tried to kiss Dudley goodbye but 
missed, because Dudley was now having a tantrum and throwing 
his cereal at the walls. ‘Little tyke,’ chortled Mr Dursley as he left 
the house. He got into his car and backed out of number four’s 
drive.


In [51]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 62.9 MB/s eta 0:00:00


In [67]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/tmp/ipython-input-67-3550734685.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [68]:

from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS



vector_store = FAISS.from_documents(documents=splitted_pages,embedding=embeddings)

In [71]:
query = "What is hogwarts?"
results = vector_store.similarity_search(query, k=5)

for i, res in enumerate(results):
    print(f"\nResult {i+1}:\n", res.page_content)


Result 1:
 Hogwarts before you can say “Quidditch”. Come on, dear.’ 
Neville, his face tear-streaked, cl utching his wrist, hobbled off 
with Madam Hooch, who had her arm around him. 
No sooner were they out of earshot than Malfoy burst into 
laughter. 
‘Did you see his face, the great lump?’ 
The other Slytherins joined in. 
‘Shut up, Malfoy,’ snapped Parvati Patil. 
‘Ooh, sticking up for Longbottom?’ said Pansy Parkinson, a 
hard-faced Slytherin girl. ‘Never thought you’d like fat little cry

Result 2:
 we go!’ 
And the school bellowed: 
 
‘Hogwarts, Hogwarts, Hoggy Warty Hogwarts, 
Teach us something please, 
Whether we be old and bald 
Or young with scabby knees, 
Our heads could do with filling 
With some interesting stuff, 
For now they’re bare and full of air, 
Dead flies and bits of fluff, 
So teach us things worth knowing, 
Bring back what we’ve forgot, 
Just do your best, we’ll do the rest, 
And learn until our brains all rot.’

Result 3:
 When a letter arrives for unhappy b